# 06 — CNN rung 2: full-data transfer check, batch/LR pick, timing

**Decision this feeds** (`docs/superpowers/specs/2026-09-09-cnn-train-rung23-design.md`):
measures wall-clock time for one full-scale nested-CV fold (needed to
budget rung 3's 5-fold × 5-repeat run against the 2026-09-16 deadline),
picks a batch size/learning rate empirically among three candidates, and
runs one leave-one-family-out transfer check (holding out the 3.895mm
family, not the largest 2.46mm family — see the spec for why). **This
notebook does not decide the gate** — only `notebooks/07_cnn_rung3.ipynb`'s
full nested CV, scored against `model.build_combat_baseline()` = 0.5290
log loss, does that.

**Nested cross-validation**: every training run below splits its training
portion again, 90/10, to pick the early-stopping epoch — the row(s) being
scored (the outer test fold, or the held-out family) are never used for
training or stopping. This avoids the optimism bias of scoring on the
same data used for model selection.

**Data handling**: this notebook loads real `.nii.gz` volumes and
row-level labels throughout, so per the AI-assistant data rule
(`README.md`) it is **[RUN ME]** — run it yourself, share back only the
printed aggregate numbers (timings, log loss values), not any per-row
output.

In [1]:
# [RUN ME] -- loads real pixel data + row-level labels. Builds the shared
# on-disk volume cache reused by every training run below and in
# notebooks/07_cnn_rung3.ipynb.
import sys
import time
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent / "src"))

import numpy as np
import pandas as pd
import torch

import cache
import config
import dataset
import evaluate
import model
import train as train_mod

labels_df = pd.read_csv(config.TRAIN_LABELS_PATH)
family_df = pd.read_csv(config.DATA_PROCESSED / "baseline_features.csv")[
    [config.UID_COLUMN, "inplane_family"]
]
labeled_df = labels_df.merge(family_df, on=config.UID_COLUMN, how="inner").reset_index(drop=True)
print("family counts:\n", labeled_df["inplane_family"].value_counts())

uids = labeled_df[config.UID_COLUMN].tolist()
labels = labeled_df[config.TARGET_COLUMN].tolist()
families = labeled_df["inplane_family"].tolist()

config_fingerprint = {
    "TARGET_SPACING": config.TARGET_SPACING,
    "CROP_SIZE_MM": config.CROP_SIZE_MM,
    "CROP_CENTER_MM": config.CROP_CENTER_MM,
    "TARGET_SHAPE": config.TARGET_SHAPE,
    "BACKGROUND_PERCENTILE": config.BACKGROUND_PERCENTILE,
    "BACKGROUND_MAX_FRACTION": config.BACKGROUND_MAX_FRACTION,
}

cache_start = time.time()
volume_cache = cache.CachedVolumeStore(
    uids, cache_dir=config.DATA_PROCESSED / "volume_cache",
    config_fingerprint=config_fingerprint,
)
cache_build_seconds = time.time() - cache_start
print(f"cache build: {cache_build_seconds:.1f}s for {len(uids)} volumes "
      f"({cache_build_seconds / len(uids) * 1000:.1f} ms/volume) -- "
      f"this cost is paid once per notebook session, not once per fold.")

family counts:
 inplane_family
2.46            528
3.895           325
2.30            207
2.398           149
~1.5-1.8         48
~3.30            39
~1.47            31
2.00             10
3.591             9
4.42              9
other(1.372)      1
other(2.129)      1
other(2.415)      1
other(2.360)      1
other(1.882)      1
other(2.127)      1
other(2.017)      1
Name: count, dtype: int64
cache build: 2634.9s for 1362 volumes (1934.6 ms/volume) -- this cost is paid once per notebook session, not once per fold.


In [2]:
# [RUN ME] (no data access itself -- defines a function used by the
# [RUN ME] cells below). num_workers=0 always: a cached dataset must not
# be handed to multiple DataLoader worker processes, each of which would
# rebuild its own copy of the cache and silently multiply wall-clock time.
def train_and_score_nested(train_uids, train_labels, train_family,
                            outer_uids, batch_size, lr, seed,
                            epochs=config.EPOCHS, patience=config.PATIENCE,
                            inner_splits=10):
    """Splits (train_uids, train_labels, train_family) 90/10 (stratified,
    `inner_splits` folds, fold 0) for early stopping, trains DatCNN,
    reloads the best checkpoint, and predicts on outer_uids -- which are
    never used for training or stopping. Returns
    (outer_probs, history, best_state); outer_probs is aligned to
    outer_uids' order (the outer loader is never shuffled)."""
    inner_train_idx, inner_val_idx = evaluate.make_folds(
        train_labels, train_family, n_splits=inner_splits, random_state=seed
    )[0]

    def subset(idxs):
        return ([train_uids[i] for i in idxs], [train_labels[i] for i in idxs])

    inner_train_uids, inner_train_labels = subset(inner_train_idx)
    inner_val_uids, inner_val_labels = subset(inner_val_idx)

    inner_train_ds = dataset.DatParkinsonDataset(inner_train_uids, inner_train_labels, load_fn=volume_cache.get)
    inner_val_ds = dataset.DatParkinsonDataset(inner_val_uids, inner_val_labels, load_fn=volume_cache.get)
    inner_train_loader = torch.utils.data.DataLoader(inner_train_ds, batch_size=batch_size, shuffle=True, num_workers=0)
    inner_val_loader = torch.utils.data.DataLoader(inner_val_ds, batch_size=batch_size, num_workers=0)

    torch.manual_seed(seed)
    net = model.build_model().to(config.DEVICE)
    optimizer = torch.optim.Adam(net.parameters(), lr=lr, weight_decay=config.WEIGHT_DECAY)
    loss_fn = torch.nn.BCEWithLogitsLoss()

    best_state, history = train_mod.train_one_fold(
        net, inner_train_loader, inner_val_loader, optimizer, loss_fn,
        epochs=epochs, patience=patience, device=config.DEVICE,
        use_amp=config.USE_AMP, seed=seed,
    )
    net.load_state_dict(best_state)

    outer_ds = dataset.DatParkinsonDataset(outer_uids, load_fn=volume_cache.get)
    outer_loader = torch.utils.data.DataLoader(outer_ds, batch_size=batch_size, num_workers=0)
    outer_probs = []
    for x, _ in outer_loader:
        outer_probs.append(model.predict(net, x))
    outer_probs = np.concatenate(outer_probs)

    return outer_probs, history, best_state